### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web or running the code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions(often a JSON schema)

2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")

response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots can talk. Let me start by recalling what I know about parrots. They\'re birds, right? Some species like macaws, cockatiels, and African grays are known for their ability to mimic human speech. But why do they do that? Is it instinctual? Learned behavior? Maybe survival?\n\nFirst, I remember that parrots are highly social animals. They live in flocks, so maybe they use vocalizations to communicate with each other. If they can mimic human speech, maybe that\'s an extension of their natural communication skills. But how does mimicking human words help them in the wild? Maybe it\'s not really useful there, but in captivity, they might learn it from humans.\n\nThen there\'s the aspect of learning. I think parrots have excellent memory and learning abilities. They might imitate sounds they hear frequently. In the wild, they might mimic other birds\' calls or environmental sounds. So, in a human environment, they pick up wo

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools=model.bind_tools([get_weather])

In [4]:
response=model_with_tools.invoke("Whats the weather like in Boston?")
print(response)

for tool_call in response.tool_calls:
    #view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there\'s no typo. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': 'e79f4wxsg', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 153, 'total_tokens': 246, 'completion_time': 0.151271797, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.006716228, 'prompt_tokens_details': None, 'queue_time': 0.158319091, 'total_time': 0.157988025}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_

### Tool Execution Loops

In [7]:
#Step 1: Model generates tool calls
messages=[{"role":"user","content":"What's the weather in Boston?"}]
ai_msg=model_with_tools.invoke(messages)
messages.append(ai_msg)

#Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result=get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.content)

The weather in Boston is sunny. A perfect day to enjoy outdoor activities! 😊


In [8]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call this function with "Boston" as the location. I\'ll make sure the parameters are correctly formatted in JSON and wrap the tool call in the required XML tags.\n', 'tool_calls': [{'id': 'pa1g40qp7', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 98, 'prompt_tokens': 153, 'total_tokens': 251, 'completion_time': 0.15788712, 'completion_tokens_details': {'reasoning_tokens': 74}, 'prompt_time': 0.006322035, 'prompt_tokens_details': None, 'queue_time': 0.050372205, 'total_time': 0.164209155}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2',